# 07 -- Merge weights and export

Merges the LoRA adapter into a full model (needed for SGLang to serve it locally, and for uploading to Hugging Face Hub). Do this once you're happy with the DPO results.

*(Uses the same `GITHUB_TOKEN` Colab secret set up in notebook 01 -- see that notebook if you haven't set it up yet.)*

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os

PROJECT_DIR = '/content/drive/MyDrive/MedAlignRL'
GITHUB_USERNAME = 'YOUR_USERNAME'   # <-- change this
GITHUB_REPO = 'MedAlignRL'         # <-- change if you named it differently

try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = None
    print("No GITHUB_TOKEN secret found. Fine if your repo is public -- if it's "
          "private this clone will fail. See the setup note above.")

if GITHUB_TOKEN:
    REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git'
else:
    REPO_URL = f'https://github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git'

if not os.path.exists(PROJECT_DIR):
    print("Cloning into Drive (first time)...")
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print("Repo already in Drive, pulling latest...")
    !cd {PROJECT_DIR} && git remote set-url origin {REPO_URL} && git fetch origin && git reset --hard origin/main

%cd {PROJECT_DIR}
!pip install -q -r requirements-colab.txt


In [ ]:
%cd {PROJECT_DIR}/src
!python merge_lora.py

### Option A -- push straight to Hugging Face Hub from here

Usually faster than downloading a few GB over your own connection and re-uploading later. Needs a Hugging Face account and a write token (huggingface.co/settings/tokens).

In [ ]:
from huggingface_hub import login, HfApi

login()  # pastes a prompt for your HF token

api = HfApi()
repo_id = 'YOUR_HF_USERNAME/MedAlignRL-dpo'  # change this
api.create_repo(repo_id, exist_ok=True)
api.upload_folder(
    folder_path='../outputs/dpo_model_merged',
    repo_id=repo_id,
)
print(f'Uploaded to https://huggingface.co/{repo_id}')

### Option B -- download the folder instead

If you'd rather run the agent pipeline locally without pulling from HF, zip and download directly.

In [ ]:
!zip -r dpo_model_merged.zip ../outputs/dpo_model_merged
!zip -r rag_index.zip ../data/rag_index
from google.colab import files
files.download('dpo_model_merged.zip')
files.download('rag_index.zip')

### Push the real results back to GitHub

This is the one place in the whole pipeline that writes back to GitHub instead of just reading from it. Only touches `outputs/eval_results.json` and `outputs/human_eval_sheet.csv` -- those are the two files `.gitignore` deliberately lets through, everything else in `outputs/` (model weights, checkpoints) stays out of git on purpose.

Needs the same `GITHUB_TOKEN` Colab secret from notebook 01, scoped with write access.

In [ ]:
%cd {PROJECT_DIR}

!git config user.email "colab@example.com"
!git config user.name "Colab Runner"

!git add outputs/eval_results.json outputs/human_eval_sheet.csv
result = !git status --porcelain
if result:
    !git commit -m "Add real eval results and human-eval ratings from Colab run"
    !git push
    print("Pushed. Check your GitHub repo -- outputs/eval_results.json should be updated.")
else:
    print("Nothing new to commit -- either these files haven't changed, or they don't exist yet "
          "(run notebooks 5 and 6 first).")

### Last thing

Numbers are in GitHub now, but they're still just raw JSON/CSV. Open the README locally, copy the real figures into the results table by hand, commit, push. That's the project done -- flip the GitHub repo to public whenever you're ready.